<a href="https://colab.research.google.com/github/supereyhap-hbtf/AI/blob/main/Scikit_Learn_with_ML_Codes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Important Laibraries:**

* **Numpy:** for Math
* **Pandas:** for Data
* **matplotlib, seaborn:** for Graph
* **tensorflow, keras:** for DL (Deep Learning)
* **Sklearn:** for Training

## **SKlearn: Scientific Kit Learn**

In [1]:
# ============================================================
# POKEMON DATASET - ENCODING & SCALING
# ============================================================
# Goal:
# Learn how to prepare a real dataset for Machine Learning.
#
# Topics we will learn:
# - Understanding the dataset
# - Numerical vs categorical data
# - Missing values
# - Train/Test Split
# - Encoding
# - Scaling
# - ColumnTransformer
# - Pipeline
# - fit() vs transform()
# - Avoiding data leakage
# ============================================================


# Import Pandas for working with the dataset
import pandas as pd

In [4]:
# ============================================================
# STEP 2: LOAD THE DATASET
# ============================================================

# Read the CSV file
df = pd.read_csv("data.csv")

# Display the first 5 rows
df.head()

,No,Name,Type1,Type2,Height,Weight,Legendary
0,1,Bulbasaur,Grass,Poison,0.7,6.9,0
1,2,Ivysaur,Grass,Poison,1.0,13.0,0
2,3,Venusaur,Grass,Poison,2.0,100.0,0
3,4,Charmander,Fire,NaN,0.6,8.5,0
4,5,Charmeleon,Fire,NaN,1.1,19.0,0


In [5]:
# ============================================================
# STEP 3: CHECK THE SIZE OF THE DATASET
# ============================================================

# shape returns:
# number of rows, number of columns

df.shape

(150, 7)

Meaning:

150 rows

7 columns

In [6]:
# ============================================================
# STEP 4: GET BASIC INFORMATION ABOUT THE COLUMNS
# ============================================================

# info() shows:
# - column names
# - number of non-null values
# - data types
# - memory usage

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   No         150 non-null    int64  
 1   Name       150 non-null    object 
 2   Type1      150 non-null    object 
 3   Type2      67 non-null     object 
 4   Height     150 non-null    float64
 5   Weight     150 non-null    float64
 6   Legendary  150 non-null    int64  
dtypes: float64(2), int64(2), object(3)
memory usage: 8.3+ KB


In [7]:
# ============================================================
# STEP 5: CHECK FOR MISSING VALUES
# ============================================================

# isnull() identifies missing values
# sum() counts how many missing values exist in each column

df.isnull().sum()

,0
No,0
Name,0
Type1,0
Type2,83
Height,0
Weight,0
Legendary,0


In [8]:
# ============================================================
# STEP 6: LOOK AT ALL COLUMN NAMES
# ============================================================

# Display all column names in our dataset
df.columns

Index(['No', 'Name', 'Type1', 'Type2', 'Height', 'Weight', 'Legendary'], dtype='object')

Now let's think about each column.

| Column      | Type             | Keep?     | Why?                                |
| ----------- | ---------------- | --------- | ----------------------------------- |
| `No`        | Numerical        | ❌         | Just an ID/number                   |
| `Name`      | Categorical/Text | ❌         | Name isn't useful for this exercise |
| `Type1`     | Categorical      | ✅         | Useful feature                      |
| `Type2`     | Categorical      | ✅         | Useful feature                      |
| `Height`    | Numerical        | ✅         | Useful feature                      |
| `Weight`    | Numerical        | ✅         | Useful feature                      |
| `Legendary` | 0/1              | 🎯 Target | What we want to predict             |

Important lesson

Not every numerical column should automatically be scaled.

For example:

No = 1, 2, 3, 4, ...

Although No is numerical, it is really an identifier.

We don't want our model thinking:
Pokemon 100

is somehow twice as important as:
Pokemon 50

So we'll remove it.

Our target is:
Legendary

We want to predict whether a Pokémon is legendary.

In [9]:
# ============================================================
# STEP 7: SEPARATE FEATURES (X) AND TARGET (y)
# ============================================================

# X = features used by the Machine Learning model
#
# We remove:
# - No     -> identifier, not a meaningful ML feature
# - Name   -> text/name, not useful for this exercise
# - Legendary -> target that we want to predict

X = df.drop(columns=["No", "Name", "Legendary"])

# y = target variable
# 0 = not legendary
# 1 = legendary

y = df["Legendary"]

# Display the first 5 rows of X
X.head()

,Type1,Type2,Height,Weight
0,Grass,Poison,0.7,6.9
1,Grass,Poison,1.0,13.0
2,Grass,Poison,2.0,100.0
3,Fire,NaN,0.6,8.5
4,Fire,NaN,1.1,19.0


In [10]:
# Display the first 5 values of our target
y.head()

,Legendary
0,0
1,0
2,0
3,0
4,0


Think of it as:

X
┌────────┬────────┬────────┬────────┐
│ Type1  │ Type2  │ Height │ Weight │
└────────┴────────┴────────┴────────┘
                 ↓
          Information

y
┌───────────┐
│ Legendary │
└───────────┘
      ↓
    Answer

In [11]:
# Let's see how many legendary and non-legendary Pokémon we have.

# ============================================================
# STEP 8: CHECK THE TARGET DISTRIBUTION
# ============================================================

# value_counts() counts how many times each value appears
print(y.value_counts())

Legendary
0    146
1      4
Name: count, dtype: int64


In [12]:
# Show the percentage of each class
print(y.value_counts(normalize=True) * 100)

Legendary
0    97.333333
1     2.666667
Name: proportion, dtype: float64


This is important because we don't want a situation where, for example:

99% = Not Legendary

1%  = Legendary

and then randomly split the data in a way that makes the test set unrepresentative.

We'll address this when we split the data.

In [13]:
# Now we're getting directly into Scaling.

# ============================================================
# STEP 9: IDENTIFY NUMERICAL FEATURES
# ============================================================

# Select columns containing numerical data
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

print("Numerical features:")
print(numeric_features)

Numerical features:
Index(['Height', 'Weight'], dtype='object')


These are the columns that could be candidates for scaling.

In [14]:
# Now the Encoding part.

# ============================================================
# STEP 10: IDENTIFY CATEGORICAL FEATURES
# ============================================================

# Select columns containing categorical/text data
categorical_features = X.select_dtypes(
    include=["object"]
).columns

print("Categorical features:")
print(categorical_features)

Categorical features:
Index(['Type1', 'Type2'], dtype='object')


So now our preprocessing plan is becoming clear:

Type1 ──┐
        ├──→ Encoding
Type2 ──┘

Height ──┐
         ├──→ Scaling
Weight ──┘

In [15]:
# Before encoding, we need to deal with missing values.

# ============================================================
# STEP 11: CHECK MISSING VALUES IN X
# ============================================================

# Count missing values in each feature
X.isnull().sum()

,0
Type1,0
Type2,83
Height,0
Weight,0


You already saw:

Type2 → 83 missing values

Now comes an important question:

What does a missing Type2 mean?

In this dataset, a Pokémon doesn't necessarily have a second type.

So:

NaN

doesn't necessarily mean:

"We don't know the Type2."

It can mean:

"This Pokémon doesn't have a second type."

Therefore, for our learning exercise, we'll replace the missing Type2 values with a category such as:

None

This is actually a good example of why understanding the dataset comes before coding.

In [16]:
# Now we split our data.
# This is extremely important.

# ============================================================
# STEP 12: SPLIT THE DATA INTO TRAINING AND TESTING DATA
# ============================================================

from sklearn.model_selection import train_test_split

# Split X and y into training and testing sets
#
# test_size=0.20 means:
# 80% -> training data
# 20% -> testing data
#
# random_state=42 makes the split reproducible
#
# stratify=y keeps the proportion of Legendary/non-Legendary
# approximately the same in both training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display the sizes
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (120, 4)
X_test : (30, 4)
y_train: (120,)
y_test : (30,)


🧠 Why did we split BEFORE preprocessing?

This is one of the most important ideas in this whole lesson.

We want:

Original Dataset
       ↓
Train/Test Split
       ↓
   ┌───────┴───────┐
   ↓               ↓
Training          Testing
   ↓               ↓
Learn             Apply

We do not want the preprocessing to learn information from the test data.

That's how we avoid data leakage.

In [17]:
# Create the Imputer
# Now let's handle the missing Type2
# We'll use:
# SimpleImputer

# ============================================================
# STEP 13: HANDLE MISSING CATEGORICAL VALUES
# ============================================================

from sklearn.impute import SimpleImputer

# Create an imputer for categorical columns
#
# strategy="constant" means:
# replace missing values with a fixed value
#
# fill_value="None" means:
# missing Type2 becomes the category "None"

categorical_imputer = SimpleImputer(
    strategy="constant",
    fill_value="None"
)

Notice something important:

We are not actually applying it yet.

We're just creating the preprocessing tool.

In [18]:
# Create OneHotEncoder
# Now we're ready for Encoding

# ============================================================
# STEP 14: CREATE THE ONE-HOT ENCODER
# ============================================================

from sklearn.preprocessing import OneHotEncoder

# Create the encoder
#
# handle_unknown="ignore" means:
# if the test data contains a category that was not present
# during training, don't produce an error.

encoder = OneHotEncoder(
    handle_unknown="ignore"
)

Why OneHotEncoder?

Because:

Fire
Water
Grass

doesn't have a natural numerical order.

We don't want:

Fire  = 0
Water = 1
Grass = 2

because that would falsely suggest:

Fire < Water < Grass

Instead, OneHotEncoder creates separate binary columns.

Conceptually:

Type1_Fire
Type1_Water
Type1_Grass

In [19]:
# Create StandardScaler
# Now the Scaling part.

# ============================================================
# STEP 15: CREATE THE STANDARD SCALER
# ============================================================

from sklearn.preprocessing import StandardScaler

# Create the scaler
#
# StandardScaler transforms numerical features so that
# they are centered around 0 and placed on a comparable scale.

scaler = StandardScaler()

We'll use it on:

Height

Weight

# **Understand the preprocessing plan**

At this point we have:

Categorical:
    Type1
    Type2
       ↓
    Imputer
       ↓
    OneHotEncoder


Numerical:
    Height
    Weight
       ↓
    StandardScaler


We could manually perform each operation.

But scikit-learn provides a much better tool:

**ColumnTransformer**

In [20]:
# Build ColumnTransformer

# ============================================================
# STEP 17: CREATE THE COLUMN TRANSFORMER
# ============================================================

from sklearn.compose import ColumnTransformer

# ColumnTransformer allows us to apply different
# preprocessing operations to different columns.

preprocessor = ColumnTransformer(
    transformers=[

        # ----------------------------------------------------
        # NUMERICAL PIPELINE
        # ----------------------------------------------------
        # Apply StandardScaler to:
        # Height and Weight

        ("num", scaler, numeric_features),


        # ----------------------------------------------------
        # CATEGORICAL PIPELINE
        # ----------------------------------------------------
        # First handle missing values,
        # then apply OneHotEncoder.

        ("cat",
         # We will temporarily use a Pipeline here
         # because categorical data needs TWO operations.
         None,
         categorical_features)
    ]
)

**Stop here for a second.**

We encountered something important.

For numerical data we only need:

StandardScaler

But categorical data needs:

Missing values
      ↓
Encoding

That means we need **Pipeline**.

In [21]:
# Create a categorical Pipeline

# ============================================================
# STEP 18: CREATE THE CATEGORICAL PIPELINE
# ============================================================

from sklearn.pipeline import Pipeline

# A Pipeline allows us to execute multiple preprocessing
# steps in a specific order.

categorical_pipeline = Pipeline(
    steps=[

        # Step 1:
        # Replace missing Type2 values with "None"
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value="None"
        )),

        # Step 2:
        # Convert categorical values into numerical columns
        ("encoder", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

Now we have:

Type1 / Type2
      ↓
   Imputer
      ↓
OneHotEncoder
      ↓
Numerical data

In [22]:
# Build the final ColumnTransformer
# Now let's replace our temporary transformer.

# ============================================================
# STEP 19: BUILD THE FINAL PREPROCESSOR
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[

        # Numerical columns:
        # Height and Weight
        # → StandardScaler

        ("num",
         StandardScaler(),
         numeric_features),


        # Categorical columns:
        # Type1 and Type2
        # → Imputer → OneHotEncoder

        ("cat",
         categorical_pipeline,
         categorical_features)
    ]
)

Now our entire preprocessing system is:

                    X
                    │
          ┌─────────┴─────────┐
          ↓                   ↓
     Numerical            Categorical
    Height/Weight          Type1/Type2
          ↓                   ↓
 StandardScaler           Imputer
                              ↓
                        OneHotEncoder
          │                   │
          └─────────┬─────────┘
                    ↓
            Processed dataset

This is the key concept you wanted to learn.

In [23]:
# Fit and Transform the Training Data
# Now we actually use the preprocessor.

# ============================================================
# STEP 20: FIT AND TRANSFORM TRAINING DATA
# ============================================================

# fit_transform() does TWO things:
#
# 1. fit:
#    Learn the required information from X_train
#
# 2. transform:
#    Apply that learned information to X_train

X_train_processed = preprocessor.fit_transform(X_train)

print("Processed training data shape:")
print(X_train_processed.shape)

Processed training data shape:
(120, 29)


**This is VERY important:**

preprocessor.fit_transform(X_train)

means:

LEARN from training data
        +
TRANSFORM training data

In [24]:
# Transform the Test Data

# ============================================================
# STEP 21: TRANSFORM TESTING DATA
# ============================================================

# IMPORTANT:
# We use transform() ONLY.
#
# We DO NOT use fit_transform() on X_test.
#
# The preprocessing rules were already learned from X_train.

X_test_processed = preprocessor.transform(X_test)

print("Processed testing data shape:")
print(X_test_processed.shape)

Processed testing data shape:
(30, 29)


Remember:

X_train
   ↓
fit_transform()
   ↓
Learn + Transform


X_test
   ↓
transform()
   ↓
Transform only

In [25]:
# Understand the Shape
# Let's compare:

# ============================================================
# STEP 22: COMPARE ORIGINAL AND PROCESSED SHAPES
# ============================================================

print("Original X_train shape:")
print(X_train.shape)

print("\nProcessed X_train shape:")
print(X_train_processed.shape)

Original X_train shape:
(120, 4)

Processed X_train shape:
(120, 29)


ou will notice that the number of columns increased.

Why?

Because:

Type1

became several columns.

And:

Type2

also became several columns.

For example:

Type1_Fire
Type1_Water
Type1_Grass
...

That's exactly what One-Hot Encoding does.

In [26]:
# See the generated feature names
# This is a very useful trick.

# ============================================================
# STEP 23: SEE THE NEW FEATURE NAMES
# ============================================================

# Get the names of the columns created after preprocessing

feature_names = preprocessor.get_feature_names_out()

# Display them
print(feature_names)

['num__Height' 'num__Weight' 'cat__Type1_Bug' 'cat__Type1_Dragon'
 'cat__Type1_Electric' 'cat__Type1_Fairy' 'cat__Type1_Fighting'
 'cat__Type1_Fire' 'cat__Type1_Ghost' 'cat__Type1_Grass'
 'cat__Type1_Ground' 'cat__Type1_Ice' 'cat__Type1_Normal'
 'cat__Type1_Poison' 'cat__Type1_Psychic' 'cat__Type1_Rock'
 'cat__Type1_Water' 'cat__Type2_Fairy' 'cat__Type2_Fighting'
 'cat__Type2_Flying' 'cat__Type2_Grass' 'cat__Type2_Ground'
 'cat__Type2_Ice' 'cat__Type2_None' 'cat__Type2_Poison'
 'cat__Type2_Psychic' 'cat__Type2_Rock' 'cat__Type2_Steel'
 'cat__Type2_Water']


You'll see something conceptually like:

num__Height

num__Weight

cat__Type1_Bug

cat__Type1_Dark

cat__Type1_Dragon

...

cat__Type2_Bug

cat__Type2_Dark

cat__Type2_Dragon

...

cat__Type2_None


This is one of the best ways to understand what the preprocessing actually did.

In [27]:
# Convert the processed data back to a DataFrame
# The result from ColumnTransformer may be a sparse matrix.
# For learning purposes, let's turn it into a Pandas DataFrame.

# ============================================================
# STEP 24: CREATE A DATAFRAME FROM THE PROCESSED DATA
# ============================================================

# Convert the processed training data into a DataFrame
#
# The columns are the feature names created by the
# preprocessing process.

X_train_processed_df = pd.DataFrame(
    X_train_processed.toarray(),
    columns=feature_names,
    index=X_train.index
)

# Display the first 5 rows
X_train_processed_df.head()

# Now you can actually see the final dataset.

,num__Height,num__Weight,cat__Type1_Bug,cat__Type1_Dragon,cat__Type1_Electric,cat__Type1_Fairy,cat__Type1_Fighting,cat__Type1_Fire,cat__Type1_Ghost,cat__Type1_Grass,...,cat__Type2_Flying,cat__Type2_Grass,cat__Type2_Ground,cat__Type2_Ice,cat__Type2_None,cat__Type2_Poison,cat__Type2_Psychic,cat__Type2_Rock,cat__Type2_Steel,cat__Type2_Water
4,-0.109467,-0.442393,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
16,-0.109467,-0.247194,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
51,-0.780649,-0.705026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
24,-0.780649,-0.673084,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
143,0.465832,0.203540,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
# Look at the scaled numerical values

# ============================================================
# STEP 25: CHECK THE SCALED NUMERICAL FEATURES
# ============================================================

# Display the transformed Height and Weight columns

X_train_processed_df[
    ["num__Height", "num__Weight"]
].head()

,num__Height,num__Weight
4,-0.109467,-0.442393
16,-0.109467,-0.247194
51,-0.780649,-0.705026
24,-0.780649,-0.673084
143,0.465832,0.203540


You should see values such as:

-0.5

 0.2

 1.3

-1.1

...

Don't worry about memorizing the values.

The important thing is:

The original Height and Weight values have been transformed by StandardScaler.

In [29]:
# Look at the encoded categorical values
# ============================================================
# STEP 26: CHECK THE ENCODED CATEGORICAL FEATURES
# ============================================================

# Display the columns generated for Type1 and Type2

categorical_feature_names = [
    column
    for column in feature_names
    if column.startswith("cat__")
]

print(categorical_feature_names)

['cat__Type1_Bug', 'cat__Type1_Dragon', 'cat__Type1_Electric', 'cat__Type1_Fairy', 'cat__Type1_Fighting', 'cat__Type1_Fire', 'cat__Type1_Ghost', 'cat__Type1_Grass', 'cat__Type1_Ground', 'cat__Type1_Ice', 'cat__Type1_Normal', 'cat__Type1_Poison', 'cat__Type1_Psychic', 'cat__Type1_Rock', 'cat__Type1_Water', 'cat__Type2_Fairy', 'cat__Type2_Fighting', 'cat__Type2_Flying', 'cat__Type2_Grass', 'cat__Type2_Ground', 'cat__Type2_Ice', 'cat__Type2_None', 'cat__Type2_Poison', 'cat__Type2_Psychic', 'cat__Type2_Rock', 'cat__Type2_Steel', 'cat__Type2_Water']


In [30]:
# Display some of the encoded categorical data

X_train_processed_df[categorical_feature_names].head()

,cat__Type1_Bug,cat__Type1_Dragon,cat__Type1_Electric,cat__Type1_Fairy,cat__Type1_Fighting,cat__Type1_Fire,cat__Type1_Ghost,cat__Type1_Grass,cat__Type1_Ground,cat__Type1_Ice,...,cat__Type2_Flying,cat__Type2_Grass,cat__Type2_Ground,cat__Type2_Ice,cat__Type2_None,cat__Type2_Poison,cat__Type2_Psychic,cat__Type2_Rock,cat__Type2_Steel,cat__Type2_Water
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
51,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
24,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
143,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


You'll see lots of:

0.0

1.0

That's the result of **One-Hot Encoding**.

In [31]:
# Understand fit()
# Now let's isolate one of the most important concepts.
# Suppose we use StandardScaler.

# ============================================================
# STEP 27: UNDERSTAND FIT()
# ============================================================

scaler_example = StandardScaler()

# fit() learns the information needed for scaling
# from the training data.

scaler_example.fit(X_train[["Height", "Weight"]])

# Display what the scaler learned

print("Means learned by the scaler:")
print(scaler_example.mean_)

print("\nStandard deviations learned by the scaler:")
print(scaler_example.scale_)

Means learned by the scaler:
[ 1.21416667 43.93      ]

Standard deviations learned by the scaler:
[ 1.04293623 56.35256368]


The scaler has l**earned information from the training data**.

It doesn't just blindly transform numbers.

It first learns what the training data looks like.

In [32]:
# Understand transform()

# ============================================================
# STEP 28: UNDERSTAND TRANSFORM()
# ============================================================

# transform() uses the information learned by fit()

scaled_example = scaler_example.transform(
    X_train[["Height", "Weight"]]
)

print(scaled_example[:5])

[[-0.10946658 -0.44239336]
 [-0.10946658 -0.24719372]
 [-0.78064856 -0.7050256 ]
 [-0.78064856 -0.67308384]
 [ 0.46583225  0.20353999]]


So:

fit()
  ↓
Learn

transform()
  ↓
Apply what was learned

And:

fit_transform()

is simply a convenient combination for the training data.

In [33]:
# The Data Leakage Rule
# Let's make this a permanent comment in your notebook.

# ============================================================
# STEP 29: IMPORTANT - AVOID DATA LEAKAGE
# ============================================================

# CORRECT:
#
# Training data:
# fit_transform()
#
# Testing data:
# transform()

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)


# WRONG:
#
# X_test_processed = preprocessor.fit_transform(X_test)
#
# Why?
# Because fit() would learn information from the test data.
#
# The test data should remain unseen during the learning process.

**Memorize this:**

Fit only on training data.

This rule is extremely important in machine learning.

In [ ]:
# Build the Complete Pipeline
# We've already learned Pipeline with the categorical columns.


# Build the Complete Pipeline

We've already learned Pipeline with the categorical columns.

But there's an even better use of Pipeline:

We can combine the entire preprocessing process with a machine-learning model.

First, let's just create the preprocessing pipeline.

Actually, our ColumnTransformer already acts as the main preprocessing stage.

We can later do:

Preprocessing
     ↓
   Model

Using:

Pipeline

In [34]:
# Add a Machine Learning Model
# Since our target is: Legendary
# we can use a simple classification model.
# Let's use Logistic Regression just to demonstrate the complete workflow.

# ============================================================
# STEP 31: CREATE A COMPLETE MACHINE LEARNING PIPELINE
# ============================================================

from sklearn.linear_model import LogisticRegression

# Create a pipeline:
#
# Step 1:
# Preprocess the data
#
# Step 2:
# Train the Logistic Regression model

model_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

In [35]:
# Train the Pipeline

# ============================================================
# STEP 32: TRAIN THE COMPLETE PIPELINE
# ============================================================

# The pipeline will automatically:
#
# 1. Fit the preprocessing on X_train
# 2. Transform X_train
# 3. Train Logistic Regression
#
# The target y_train is used only by the model.

model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['Height', 'Weight'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='None',
                                                                                 strategy='constant')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Type1', 'Type2'], dtype='object'))])),
                ('model', LogisticRegression(max_iter=1000))])

This is beautiful because you don't have to manually write:

imputer

encoder

scaler

transform

model

every time.

The Pipeline handles the sequence.

In [36]:
# Make Predictions

# ============================================================
# STEP 33: MAKE PREDICTIONS
# ============================================================

# The pipeline automatically preprocesses X_test
# using the rules learned from X_train,
# then makes predictions.

y_pred = model_pipeline.predict(X_test)

print(y_pred)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [37]:
# Evaluate the Model
# We are not studying Machine Learning evaluation deeply yet, but let's see the result.

# ============================================================
# STEP 34: CHECK MODEL ACCURACY
# ============================================================

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.9666666666666667


You may get a result that changes slightly depending on the split/version.

**Don't focus on getting a high score here.**

Our objective was preprocessing.

In [38]:
# Test the Pipeline with New Pokémon
# This is a very useful final exercise.
# Let's create new Pokémon information:

# ============================================================
# STEP 35: TEST WITH NEW DATA
# ============================================================

# Create new Pokémon information
#
# IMPORTANT:
# We do NOT include:
# - No
# - Name
# - Legendary
#
# because the model should predict Legendary.

new_pokemon = pd.DataFrame({
    "Type1": ["Dragon"],
    "Type2": ["Flying"],
    "Height": [2.5],
    "Weight": [100.0]
})

# Ask the trained pipeline to make a prediction

prediction = model_pipeline.predict(new_pokemon)

print("Prediction:", prediction)

Prediction: [0]


The result will be:

0

or:

1

Conceptually:

0 → Not Legendary

1 → Legendary

The important part is that you gave the model raw data.

The Pipeline automatically performed:

     New Pokémon
         ↓
 Missing-value handling
         ↓
      Encoding
         ↓
      Scaling
         ↓
 Logistic Regression
         ↓
     Prediction

# Your Complete Mental Model
At this point, I want you to be able to look at the Pokémon dataset and think:

                POKÉMON DATASET
                        │
                        ↓
               Understand columns
                        │
                        ↓
                Identify target
                        │
                        ↓
                Separate X and y
                        │
                        ↓
               Train/Test Split
                        │
             ┌──────────┴──────────┐
             ↓                     ↓
       Numerical data        Categorical data
       Height / Weight        Type1 / Type2
             ↓                     ↓
       StandardScaler          Imputer
                                   ↓
                             OneHotEncoder
             │                     │
             └──────────┬──────────┘
                        ↓
                    Processed X
                        │
                        ↓
                     ML Model
                        │
                        ↓
                    Prediction

# ⭐ Your final reference table

Keep this table in your notes:

| Problem                                    | Tool                |
| ------------------------------------------ | ------------------- |
| Missing values                             | `SimpleImputer`     |
| Categorical data                           | `OneHotEncoder`     |
| Ordered categorical data                   | `OrdinalEncoder`    |
| Numerical scaling                          | `StandardScaler`    |
| Scale to 0–1                               | `MinMaxScaler`      |
| Different processing for different columns | `ColumnTransformer` |
| Multiple steps in sequence                 | `Pipeline`          |
| Split data                                 | `train_test_split`  |
| Training preprocessing                     | `fit_transform()`   |
| Testing preprocessing                      | `transform()`       |


And the most important rule:

                    TRAINING DATA
                         ↓
                    fit_transform
                         ↓
                   Learn + Transform


                      TEST DATA
                         ↓
                      transform
                         ↓
                    Transform only

# 🎯 What you've actually learned

You have now gone from a raw CSV:

data.csv

to a proper ML preprocessing workflow:

  Raw Dataset
     ↓
Understand Data
     ↓
Remove irrelevant columns
     ↓
Separate X / y
     ↓
Identify numerical/categorical data
     ↓
Train/Test Split
     ↓
Handle missing values
     ↓
One-Hot Encoding
     ↓
Standard Scaling
     ↓
ColumnTransformer
     ↓
Pipeline
     ↓
Machine Learning Model
     ↓
Prediction

And this is why I think the Pokémon dataset was a **good choice for your learning objective:** it gives us numerical data, categorical data, missing categorical values, a binary target, and enough variety to demonstrate the complete preprocessing workflow.

**One important distinction**

We used StandardScaler here because we are learning scaling. That does not mean every ML algorithm always needs scaling. For example, many tree-based models such as Decision Trees and Random Forests generally don't require feature scaling. We'll learn when scaling is actually necessary when you get to the individual algorithms.

**Your next step should be to keep this Colab notebook.** Don't just read it once. Try changing one thing at a time—remove StandardScaler, remove OneHotEncoder, change test_size, inspect the feature names, and see what changes. That's where the concepts will really stick.